# Estimate Upstream (Embodied) Emissions for GM-SEUS Arrays
NOTE: Requires the getGeneration model inputs (modeled mount) and, for the net offset section, the getGridEmissions long offset table.

This script estimates the one-time upstream greenhouse gas emissions (t CO2e) of building each GM-SEUS ground-mounted photovoltaic array: mining, materials, and manufacturing of the modules, plus the balance of system (mounting structures, trackers, inverters, transformers, cables), site preparation, and construction. They are booked in the installation year. All GM-SEUS arrays are ground-mounted commercial or utility-scale systems; no rooftop systems are included.

For each array: **upstream (t CO2e) = DC capacity (MW) x [module factor (kg CO2e/kW) + balance of system factor (kg CO2e/kW)]**, computed for a low, central, and high bound.

| Component | Depends on | Low | Central | High | Source |
|---|---|---|---|---|---|
| **c-Si module** (kg CO2e/kW, at the reference production year) | module type, manufacturing year | 591 (2022) | 753 (2020) | 810 (2020) | IEA PVPS Task 12 (2023); Fthenakis, Leccisi & Sinha (2024); Mueller et al. (2021) |
| **c-Si trend over time** (reduction per doubling of global cumulative PV capacity) | manufacturing year | 14.2% | 16.5% | 23.6% | Louwen et al. (2016) |
| **CdTe module** (kg CO2e/kW, held constant over time) | module type | 150 | 271 | 295 | Fthenakis, Leccisi & Sinha (First Solar Series 7, 6, 4) |
| **Balance of system** (kg CO2e per m2 of module area) | mount | fixed 42.3, tracker 47.4 | same | same | Fthenakis, Leccisi & Sinha (utility-scale ground mount) |

**How each value is derived**
* **c-Si low (591)**: IEA PVPS Task 12 fact sheet (2023 update, reference production year 2022) gives mono-Si **module** emissions of 20.2 g CO2e per kWh for a reference system yielding 976 kWh/kW/yr over 30 years. Converting back to capacity: 20.2 x 976 x 30 / 1000 = 591 kg CO2e/kW. Only the module share is used; the IEA reference system is rooftop, but module manufacturing emissions do not depend on how the module is later mounted, and the rooftop system's mounting is not used.
* **c-Si central (753)**: Fthenakis, Leccisi & Sinha report First Solar Series 6 CdTe modules at 271 kg CO2e/kW and state this is about 36% of single-crystalline Si modules made in China (20.5% efficiency), so 271 / 0.36 = 753 kg CO2e/kW. Most modules installed in the US were made in China or Southeast Asia.
* **c-Si high (810)**: Mueller et al. (2021): glass-glass c-Si modules produced in China, 810 kg CO2e/kW (vs. 580 Germany, 480 EU), driven by the coal-heavy manufacturing grid.
* **Trend**: Louwen et al. (2016) find c-Si production GHG falls 16.5 +/- 2.3% (poly-Si) and 23.6 +/- 1.9% (mono-Si) per doubling of cumulative installed PV capacity. Central uses poly-Si (the dominant technology for most installations through ~2017); low and high use poly-Si minus one standard deviation and mono-Si. Cumulative global capacity is from IRENA (via Our World in Data).
* **CdTe**: First Solar Series 7 (150, cut-off model, 2022-2023 US/India production), Series 6 (271), and Series 4 (295). Held constant over time (CdTe is ~3% of GM-SEUS arrays).
* **Balance of system**: utility-scale ground-mount BOS (mounting, supporting structures, trackers and motors, inverters, transformers, cables, site preparation, operation and maintenance) per m2 of module: 42.3 (fixed latitude tilt) and 47.4 (1-axis tracker) kg CO2e/m2. Converted per kW using the array's module efficiency (m2 per kW = 1 / efficiency). The authors note these come from installations with older, smaller modules, so current BOS is likely somewhat lower; the same value is used in all three bounds.

**Assumptions**
1. **Timing**: upstream emissions are a one-time emission in the installation year. Modules are assumed manufactured `mfgLagYears` (1) year before installation; the trend uses global cumulative capacity in the manufacturing year.
2. **Trend limits**: before `curveStartYear` (2005) the c-Si factor is held at its 2005 value; after the last capacity data year it is held at the last value.
3. **Bound direction**: before a bound's reference year, a lower learning rate gives lower emissions; after it, a higher one does. So the low bound uses the low rate before its reference year and the high rate after it (and vice versa for the high bound), keeping low <= central <= high.
4. **Module type**: `cdte` uses the CdTe factors; all other PV types (`c-si`, `cigs`, `a-si`, `thin-film`, `thin-film-other`) use c-Si factors (IEA PVPS reports CIS/CIGS emissions close to mono-Si). CSP arrays are excluded.
5. **Mount**: from the getGeneration modeled mount (`pvMount`: fixed, single_axis, dual_axis); dual-axis uses the tracker BOS value.
6. **Excluded**: end-of-life (recycling and disposal; generally small, and recycling credits can offset it), inverter replacement beyond what the BOS data include, and land-use change.
7. **GWP**: 100-year GWPs as used by each source LCA (IPCC AR5/AR6 depending on source).

**Outputs** (in `Data/Derived/upstreamEmissions`)
* **GMSEUS_UpstreamEmissions**: `arrayID`, `instYr`, `mfgYr`, `modType`, `moduleFamily`, `pvMount`, `capMWDC`, `effUsed`, `moduleKgKW_low|central|high`, `bosKgKW`, `upstreamKgKW_low|central|high`, `tCO2eUpstream_low|central|high` (-9999 for arrays not estimated, e.g., CSP).
* **GMSEUS_NetOffset_<weather source>**: per array, lifetime grid offset for each version (average, marginal long-run, marginal short-run), net offset after central upstream emissions, and simple carbon payback time (central upstream / operating-year-1 offset).

**References**
* IEA PVPS Task 12 (2023). Environmental life cycle assessment of electricity from PV systems, fact sheet update 2023 (v2.0). No DOI. https://iea-pvps.org/wp-content/uploads/2024/05/Slides_IEA-PVPS-T12_Fact-Sheet-update-2023_v2.0.pdf
* Fthenakis V., Leccisi E., Sinha P. (2024). Updated sustainability status of cadmium telluride thin-film photovoltaic systems and projections. Progress in Photovoltaics: Research and Applications 32(12), 846-863. https://doi.org/10.1002/pip.3837
* Mueller A., Friedrich L., Reichel C., Herceg S., Mittag M., Neuhaus D.H. (2021). A comparative life cycle assessment of silicon PV modules: Impact of module design, manufacturing location and inventory. Solar Energy Materials and Solar Cells 230, 111277. https://doi.org/10.1016/j.solmat.2021.111277
* Louwen A., van Sark W., Faaij A., Schropp R. (2016). Re-assessment of net energy production and greenhouse gas emissions avoidance after 40 years of photovoltaics development. Nature Communications 7, 13728. https://doi.org/10.1038/ncomms13728
* IRENA (2026). Renewable Energy Statistics, processed by Our World in Data (installed solar capacity). No DOI. https://ourworldindata.org/grapher/installed-solar-pv-capacity


# Import Libraries and Variables

In [ ]:
# Import libraries
import os
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import figureTheme as ft # shared figure theme (utils/figureTheme.py)
ft.apply()

In [ ]:
# Set folder paths
wd = r'S:\Users\stidjaco\R_files\BigPanel'
downloaded_path = os.path.join(wd, r'Data\Downloaded')
derived_path = os.path.join(wd, r'Data\Derived')
lca_path = os.path.join(downloaded_path, r'LCA') # downloaded inputs for upstream emissions
upstream_path = os.path.join(derived_path, r'upstreamEmissions')
generation_path = os.path.join(derived_path, r'generation') # getGeneration outputs
carbon_path = os.path.join(derived_path, r'carbonOffset') # getGridEmissions outputs
figure_path = os.path.join(wd, r'Figures')
genWeatherSource = 'pvgis' # which getGeneration / getGridEmissions run to use

# Set GM-SEUS array path (same file as getGeneration and getGridEmissions)
gmseusArraysPath = os.path.join(downloaded_path, r'SolarDB\GMSEUS_v2_1\SHP\GMSEUS_Arrays_Final_2025_v2_1.shp')

# Set input paths: getGeneration model inputs (modeled mount), getGridEmissions long offsets (net offset section), global cumulative PV capacity (downloaded below)
genInputs_path = os.path.join(generation_path, r'GMSEUS_Generation_ModelInputs_' + genWeatherSource + '.csv')
offsetLong_path = os.path.join(carbon_path, r'GMSEUS_GridOffset_Annual_Long_' + genWeatherSource + '.csv')
globalCap_path = os.path.join(lca_path, r'OWID_installed_solar_capacity.csv')

# Set output paths
upstreamOut_path = os.path.join(upstream_path, r'GMSEUS_UpstreamEmissions.csv')
netOffset_path = os.path.join(upstream_path, r'GMSEUS_NetOffset_' + genWeatherSource + '.csv')

# Make folders if they do not exist
os.makedirs(lca_path, exist_ok=True)
os.makedirs(upstream_path, exist_ok=True)
os.makedirs(figure_path, exist_ok=True)

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Emission factors (sources and derivations in the markdown at top)

bounds = ['low', 'central', 'high']

# c-Si module factor (kg CO2e per kW DC, cradle-to-gate) at a reference production year
cSiModule = {
    'low':     {'kgKW': 20.2 * 976 * 30 / 1000, 'refYear': 2022}, # IEA PVPS Task 12 fact sheet 2023: mono-Si module 20.2 g/kWh x 976 kWh/kW/yr x 30 yr
    'central': {'kgKW': 271 / 0.36, 'refYear': 2020},           # Fthenakis, Leccisi & Sinha: CdTe Series 6 (271) is ~36% of China-made sc-Si
    'high':    {'kgKW': 810, 'refYear': 2020},                  # Mueller et al. 2021: China-made glass-glass c-Si
}

# c-Si learning rate: fractional reduction in production GHG per doubling of global cumulative PV capacity (Louwen et al. 2016)
learningRate = {'low': 0.165 - 0.0231, 'central': 0.165, 'high': 0.236} # poly-Si -1 SD, poly-Si, mono-Si

# CdTe module factor (kg CO2e per kW DC), constant over time (Fthenakis, Leccisi & Sinha: First Solar Series 7, 6, 4)
cdteModule = {'low': 150, 'central': 271, 'high': 295}

# Balance of system, utility-scale ground mount (kg CO2e per m2 of module area; Fthenakis, Leccisi & Sinha, Table 6). Same value in all bounds.
bosKgM2 = {'fixed': 42.3, 'single_axis': 47.4, 'dual_axis': 47.4}

# Module type -> factor family (CSP not estimated)
moduleFamilyMap = {'cdte': 'cdte', 'c-si': 'c-si', 'cigs': 'c-si', 'a-si': 'c-si', 'thin-film': 'c-si', 'thin-film-other': 'c-si'}

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Trend settings

mfgLagYears = 1 # modules manufactured this many years before installation
curveStartYear = 2005 # c-Si factor held at this year's value for earlier manufacturing years

# Global cumulative installed solar capacity (GW), IRENA via Our World in Data (World row)
globalCapUrl = 'https://ourworldindata.org/grapher/installed-solar-pv-capacity.csv?v=1&csvType=full&useColumnShortNames=true'
globalCapCol = 'solar__total_gw'

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Data conventions

noData = -9999
arrayIDcol = 'arrayID'

# Helper Functions

In [ ]:
# A function to download a file (skips if already downloaded)
def downloadFile(url, outPath):
    if os.path.exists(outPath):
        print('Exists, skipping:', outPath)
        return outPath
    r = requests.get(url, timeout=300, headers={'User-Agent': 'Mozilla/5.0 (research data download script)'})
    r.raise_for_status()
    with open(outPath, 'wb') as f:
        f.write(r.content)
    print('Downloaded:', outPath)
    return outPath

# A function to get the c-Si module factor (kg CO2e/kW) for manufacturing years and one bound, scaled from the bound's reference year by the learning curve:
# factor(y) = factor(ref) x (C(y) / C(ref)) ^ log2(1 - LR), with C = global cumulative capacity. Low/high learning rates swap after the reference year (keeps bounds ordered).
def cSiModuleFactor(mfgYears, bound, cumCap):
    ref = cSiModule[bound]
    years = np.clip(np.asarray(mfgYears), curveStartYear, cumCap.index.max())
    capRatio = cumCap.reindex(years).values / cumCap.loc[ref['refYear']]
    lrBefore = learningRate[bound]
    lrAfter = {'low': learningRate['high'], 'central': learningRate['central'], 'high': learningRate['low']}[bound]
    lr = np.where(years <= ref['refYear'], lrBefore, lrAfter)
    return ref['kgKW'] * capRatio ** np.log2(1 - lr)

# Download Data
Global cumulative installed solar capacity (IRENA, processed by Our World in Data) drives the c-Si trend. Emission factors are literature values set in the settings cell.

In [ ]:
# Download global cumulative solar capacity and keep the World series (GW by year)
downloadFile(globalCapUrl, globalCap_path)
globalCap = pd.read_csv(globalCap_path)
cumCap = globalCap[globalCap['entity'] == 'World'].set_index('year')[globalCapCol]
print('Global cumulative solar capacity (GW):', cumCap.loc[[2000, 2005, 2010, 2015, 2020, cumCap.index.max()]].round(1).to_dict())

# Load GM-SEUS Arrays and Assign Inputs

In [ ]:
# Call GM-SEUS array attributes (geometry not needed)
arrays = gpd.read_file(gmseusArraysPath, ignore_geometry=True, columns=[arrayIDcol, 'instYr', 'capMWDC', 'modType', 'effInit', 'mount'])
arrays = arrays.replace(noData, np.nan)
arrays['modType'] = arrays['modType'].str.lower()

# Modeled mount from getGeneration (fixed, single_axis, dual_axis)
genInputs = pd.read_csv(genInputs_path, usecols=[arrayIDcol, 'pvMount'])
arrays = arrays.merge(genInputs, on=arrayIDcol, how='left')

# Module family and manufacturing year
arrays['moduleFamily'] = arrays['modType'].map(moduleFamilyMap)
arrays['mfgYr'] = arrays['instYr'] - mfgLagYears

# Module efficiency for BOS area: GM-SEUS effInit, filled with the median for the installation year where missing
arrays['effUsed'] = arrays['effInit'].where(arrays['effInit'] > 0)
arrays['effUsed'] = arrays['effUsed'].fillna(arrays.groupby('instYr')['effUsed'].transform('median')).fillna(arrays['effUsed'].median())

# Arrays estimated: PV module type, capacity, installation year, and mount available
arrays['upEstimated'] = arrays['moduleFamily'].notna() & arrays['capMWDC'].gt(0) & arrays['instYr'].notna() & arrays['pvMount'].notna()
print('Arrays:', len(arrays), '| estimated:', arrays['upEstimated'].sum(), '| not estimated:', (~arrays['upEstimated']).sum())
print(arrays.loc[arrays['upEstimated'], 'moduleFamily'].value_counts().to_dict(), arrays.loc[arrays['upEstimated'], 'pvMount'].value_counts().to_dict())

# Calculate Upstream Emissions

In [ ]:
est = arrays['upEstimated']
isCdte = arrays['moduleFamily'] == 'cdte'

# Module factor (kg CO2e/kW) per bound: c-Si from the learning curve at the manufacturing year, CdTe constant
for b in bounds:
    arrays['moduleKgKW_' + b] = np.nan
    arrays.loc[est & ~isCdte, 'moduleKgKW_' + b] = cSiModuleFactor(arrays.loc[est & ~isCdte, 'mfgYr'].astype(int), b, cumCap)
    arrays.loc[est & isCdte, 'moduleKgKW_' + b] = cdteModule[b]

# Balance of system factor (kg CO2e/kW) = kg CO2e per m2 of module / module efficiency (m2 per kW = 1 / efficiency at 1 kW/m2)
arrays['bosKgKW'] = arrays['pvMount'].map(bosKgM2) / arrays['effUsed']
arrays.loc[~est, 'bosKgKW'] = np.nan

# Total factor and array upstream emissions (t CO2e = MW x kg/kW)
for b in bounds:
    arrays['upstreamKgKW_' + b] = arrays['moduleKgKW_' + b] + arrays['bosKgKW']
    arrays['tCO2eUpstream_' + b] = arrays['capMWDC'] * arrays['upstreamKgKW_' + b]

# Checks: bounds ordered, no missing values for estimated arrays
assert (arrays.loc[est, 'upstreamKgKW_low'] <= arrays.loc[est, 'upstreamKgKW_central'] + 1e-9).all() and (arrays.loc[est, 'upstreamKgKW_central'] <= arrays.loc[est, 'upstreamKgKW_high'] + 1e-9).all()
assert arrays.loc[est, ['upstreamKgKW_low', 'upstreamKgKW_central', 'upstreamKgKW_high']].notna().all().all()

# Summary: fleet totals and factors by installation period
print('Fleet upstream emissions (Mt CO2e): low', round(arrays['tCO2eUpstream_low'].sum() / 1e6, 2), '| central', round(arrays['tCO2eUpstream_central'].sum() / 1e6, 2), '| high', round(arrays['tCO2eUpstream_high'].sum() / 1e6, 2))
print('Capacity-weighted central factor (kg CO2e/kW):', round(arrays.loc[est, 'tCO2eUpstream_central'].sum() / arrays.loc[est, 'capMWDC'].sum()))
period = pd.cut(arrays['instYr'], [0, 2010, 2015, 2020, 2025], labels=['<=2010', '2011-2015', '2016-2020', '2021-2025'])
print(arrays[est].groupby([period[est], 'moduleFamily'], observed=True)[['moduleKgKW_low', 'moduleKgKW_central', 'moduleKgKW_high', 'bosKgKW']].median().round(0).to_string())

In [ ]:
# Plot: upstream factor (t CO2e per kW DC) by installation year: c-Si module low-high band and central line, module + hardware (central) by mount, CdTe module + hardware
years = np.arange(2005, int(arrays['instYr'].max()) + 1)
fig, ax = plt.subplots(figsize=(ft.figWidth['single'], 66 * ft.mm))
ax.fill_between(years, cSiModuleFactor(years - mfgLagYears, 'low', cumCap) / 1e3, cSiModuleFactor(years - mfgLagYears, 'high', cumCap) / 1e3, color=ft.upstreamColor, alpha=0.18, lw=0, label='c-Si module, low\u2013high')
ax.plot(years, cSiModuleFactor(years - mfgLagYears, 'central', cumCap) / 1e3, color=ft.upstreamColor, lw=1.1, label='c-Si module, central')
for mount, ls in [('fixed', (0, (3, 2))), ('single_axis', '-')]:
    med = arrays[est & ~isCdte & (arrays['pvMount'] == mount)].groupby('instYr')['upstreamKgKW_central'].median()
    med = med[med.index >= years.min()]
    ax.plot(med.index, med / 1e3, color=ft.upstreamDark, lw=0.9, ls=ls, label='c-Si module + hardware, ' + ft.mountLabels[mount].lower() + ' (median array)')
cdte = arrays[est & isCdte].groupby('instYr')['upstreamKgKW_central'].median()
ax.plot(cdte.index, cdte / 1e3, ls='none', marker='o', ms=2.2, mfc='white', mec=ft.upstreamColor, mew=0.6, label='CdTe module + hardware (median array)')
ax.set_xlabel('Installation year'); ax.set_ylabel('Upstream emissions (t CO$_2$e kW$_{DC}^{-1}$)'); ax.set_ylim(bottom=0)
ax.xaxis.get_major_locator().set_params(integer=True)
ax.legend(loc='upper right', fontsize=6)
ft.saveFig(fig, figure_path, 'upstreamFactors_byInstYr')
plt.show()

# Net Offset and Carbon Payback
Combines central upstream emissions with the lifetime grid offset from getGridEmissions (all three versions). Carbon payback (years) = central upstream / operating-year-1 offset, a simple approximation that ignores later-year changes in grid rates and degradation.

In [ ]:
# Lifetime and year-1 grid offsets per array (t CO2e) for each version
offsetLong = pd.read_csv(offsetLong_path)
versions = {'Avg': 'tCO2e', 'MarginalLR': 'tCO2eMarginalLR', 'MarginalSR': 'tCO2eMarginalSR'}
lifetime = offsetLong.groupby(arrayIDcol)[list(versions.values())].sum().rename(columns={c: 'lifetimeOffset' + v for v, c in versions.items()})
yr1 = offsetLong[offsetLong['opYr'] == 1].set_index(arrayIDcol)[list(versions.values())].rename(columns={c: 'yr1Offset' + v for v, c in versions.items()})

# Net offset (lifetime offset - central upstream) and payback years, per version
netOffset = arrays.loc[est, [arrayIDcol, 'instYr', 'capMWDC', 'tCO2eUpstream_low', 'tCO2eUpstream_central', 'tCO2eUpstream_high']].set_index(arrayIDcol).join(lifetime, how='inner').join(yr1)
for v in versions:
    netOffset['netOffset' + v] = netOffset['lifetimeOffset' + v] - netOffset['tCO2eUpstream_central']
    netOffset['paybackYrs' + v] = netOffset['tCO2eUpstream_central'] / netOffset['yr1Offset' + v]

# Summary
print('Arrays with both upstream and offset:', len(netOffset))
summary = pd.DataFrame({v: {'lifetime offset (Mt)': netOffset['lifetimeOffset' + v].sum() / 1e6, 'upstream central (Mt)': netOffset['tCO2eUpstream_central'].sum() / 1e6,
                            'net offset (Mt)': netOffset['netOffset' + v].sum() / 1e6, 'upstream / lifetime offset': netOffset['tCO2eUpstream_central'].sum() / netOffset['lifetimeOffset' + v].sum(),
                            'payback median (yr)': netOffset['paybackYrs' + v].median()} for v in versions})
print(summary.round(2).to_string())

# Export

In [ ]:
# Export per-array upstream emissions (-9999 for arrays not estimated) and net offsets
outCols = [arrayIDcol, 'instYr', 'mfgYr', 'modType', 'moduleFamily', 'pvMount', 'capMWDC', 'effUsed'] + ['moduleKgKW_' + b for b in bounds] + ['bosKgKW'] + \
          ['upstreamKgKW_' + b for b in bounds] + ['tCO2eUpstream_' + b for b in bounds]
upstreamOut = arrays[outCols].copy()
numCols = upstreamOut.columns.drop([arrayIDcol, 'modType', 'moduleFamily', 'pvMount'])
upstreamOut[numCols] = upstreamOut[numCols].round(3)
upstreamOut.loc[~est, numCols.drop(['instYr', 'capMWDC'])] = noData
upstreamOut.fillna(noData).to_csv(upstreamOut_path, index=False)
netOffset.round(3).reset_index().to_csv(netOffset_path, index=False)
print('Exported to:', upstream_path)